## ERA5 Wind Data

Extract Wind point from ERA5 GRIB dataset

## Load GRIB data

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import os

data_dir = "data"
output_dir = "output"

file_u = os.path.join(data_dir, "u10_2025.grib")
file_v = os.path.join(data_dir, "v10_2025.grib")

ds_u = xr.open_dataset(file_u, engine="cfgrib")
ds_v = xr.open_dataset(file_v, engine="cfgrib")

## Extracting 1 wind point location

Choose coordinate location inside GRIB dataset

In [ ]:
lat_pt = -5.48394
lon_pt = 107.00259

Extraction point using built-in xarray library methods

In [ ]:
u = ds_u['u10'].sel(latitude=lat_pt, longitude=lon_pt, method="nearest")
v = ds_v['v10'].sel(latitude=lat_pt, longitude=lon_pt, method="nearest")

## Wind speed & direction calculation

In [ ]:
speed = np.sqrt(u**2 + v**2)

## Wind direction (ERA5 explanation)
https://confluence.ecmwf.int/pages/viewpage.action?pageId=133262398
- According to ECMWF documentation, wind direction in meteorology is defined as the direction **from which the wind is blowing**
  Example:
  0 deg = from North,
  90 deg = from east,
  180 deg = from South,
  270 deg = from west,
- ERA5 provides wind data as u and v components, which represent the vector of wind motion
- These components describe the direction **towards which the wind is moving**, NOT the meteorological wind directions
- A conversion is required to transform u and v components into meteorological before further analysis (e.g., windrose)

The Formula:

In [ ]:
direction = (270 - np.degrees(np.arctan2(v, u))) % 360

## Export time series csv

speed (m/s),
direction (degree)

In [ ]:
df = pd.DataFrame({
    "time": u['time'].values,
    "speed": speed.values,
    "direction": direction.values
})

df.to_csv(os.path.join(output_dir, "wind_timeseries.csv"), index=False)

df.head()

## 16 wind direction bin classificaton

In [ ]:
dirs = ["N","NNE","NE","ENE","E","ESE","SE","SSE",
        "S","SSW","SW","WSW","W","WNW","NW","NNW"]

bins = np.linspace(0, 360, 17)

df['dir_class'] = pd.cut(df['direction'], bins=bins, labels=dirs, include_lowest=True)

# Wind count statistic

In [ ]:
dir_count = df['dir_class'].value_counts().sort_index()
dir_percent = dir_count / len(df) * 100

df_stat = pd.DataFrame({
    "count": dir_count,
    "percent": dir_percent
})

df_stat = df_stat.reset_index()
df_stat.columns = ["direction", "count", "percent"]

df_stat["percent"] = df_stat["percent"].round(2)

df_stat.to_csv(os.path.join(output_dir, "wind_direction_stats.csv"), index=False)

df_stat

## Windrose plot

In [ ]:
import matplotlib.pyplot as plt
from windrose import WindroseAxes

ax = WindroseAxes.from_ax()
ax.bar(df['direction'], df['speed'],
       normed=True,
       opening=0.8,
       edgecolor='white')

ax.set_legend(title="Wind Speed (m/s)")
plt.title("Windrose - ERA5")

plt.savefig(os.path.join(output_dir, "windrose.png"), dpi=300)

plt.show()